In [10]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv

In [11]:
load_dotenv("../.env")
data_path = os.getenv("Data_path_featured")

In [12]:
df=pd.read_csv(data_path)
df.head()

,CustomerID,SeniorCitizen,Partner,Dependents,Tenure,InternetService,OnlineSecurity,TechSupport,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,AvgMonthlySpend,TenureBucket
0,7590-VHVEG,0,Yes,No,1,DSL,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,29.850000,0-1yr
1,5575-GNVDE,0,No,No,34,DSL,Yes,No,One year,No,Mailed check,56.95,1889.50,0,55.573529,2-4yr
2,3668-QPYBK,0,No,No,2,DSL,Yes,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,54.075000,0-1yr
3,7795-CFOCW,0,No,No,45,DSL,Yes,Yes,One year,No,Bank transfer,42.30,1840.75,0,40.905556,2-4yr
4,9237-HQITU,0,No,No,2,Fiber optic,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,75.825000,0-1yr


In [13]:
x=df.drop(['Churn','CustomerID'],axis=1)
y=df["Churn"]

In [14]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42,stratify=y)

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler

In [16]:
x.columns

Index(['SeniorCitizen', 'Partner', 'Dependents', 'Tenure', 'InternetService',
       'OnlineSecurity', 'TechSupport', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend',
       'TenureBucket'],
      dtype='object')

In [17]:
binary_colm = ['SeniorCitizen','Partner','Dependents','OnlineSecurity','TechSupport','PaperlessBilling']
multi_cat_colm = ['InternetService', 'PaymentMethod', 'Contract', 'TenureBucket']
numerical_colm = ['Tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend']

In [18]:
binary_pipe = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))])

multi_cat_pipe = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))])

numerical_pipe = Pipeline(steps=[
    ('scaler', RobustScaler())])

In [19]:
preprocessor = ColumnTransformer(transformers=[
    ('bin', binary_pipe, binary_colm),
    ('multi', multi_cat_pipe, multi_cat_colm),
    ('num', numerical_pipe, numerical_colm)])

In [20]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

In [21]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum())}

In [22]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    scores = cross_validate(pipe, x_train, y_train, cv=cv,scoring=['roc_auc', 'f1', 'recall', 'precision'])
    print(f"\n--- {name} ---")
    for k, v in scores.items():
        if 'test' in k:
            print(f"{k}: {v.mean():.3f} (+/- {v.std():.3f})")


--- Logistic Regression ---
test_roc_auc: 0.844 (+/- 0.011)
test_f1: 0.622 (+/- 0.024)
test_recall: 0.792 (+/- 0.037)
test_precision: 0.513 (+/- 0.020)

--- Random Forest ---
test_roc_auc: 0.823 (+/- 0.012)
test_f1: 0.539 (+/- 0.024)
test_recall: 0.465 (+/- 0.021)
test_precision: 0.640 (+/- 0.031)

--- XGBoost ---
test_roc_auc: 0.820 (+/- 0.010)
test_f1: 0.589 (+/- 0.012)
test_recall: 0.649 (+/- 0.013)
test_precision: 0.539 (+/- 0.015)


In [23]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'classifier__max_depth': [3,4,5,6,7],
    'classifier__learning_rate': [0.01,0.05,0.1,0.2],
    'classifier__n_estimators': [100,200,300,500],
    'classifier__min_child_weight': [1,3,5],
    'classifier__subsample': [0.7,0.8,0.9,1.0],
    'classifier__colsample_bytree': [0.7,0.8,0.9,1.0],}

xgb_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=42, eval_metric='logloss',
                                  scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()))])

search = RandomizedSearchCV(xgb_pipe, param_dist, n_iter=50, scoring='roc_auc',cv=cv, random_state=42, n_jobs=-1)
search.fit(x_train, y_train)

print(search.best_params_)
print('Best CV ROC-AUC:', search.best_score_)

best_pipe = search.best_estimator_

{'classifier__subsample': 0.9, 'classifier__n_estimators': 500, 'classifier__min_child_weight': 1, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.01, 'classifier__colsample_bytree': 0.7}
Best CV ROC-AUC: 0.8483981106489296


In [25]:
from sklearn.metrics import precision_recall_curve, classification_report,roc_auc_score,confusion_matrix


y_proba = best_pipe.predict_proba(x_test)[:, 1]
prec, rec, thresh = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
best_t = thresh[f1_scores.argmax()]
print('Best threshold:', round(best_t, 3))

y_pred_tuned = (y_proba >= best_t).astype(int)
print(classification_report(y_test, y_pred_tuned))
print('ROC-AUC:', round(roc_auc_score(y_test, y_proba), 4))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_tuned))

Best threshold: 0.568
              precision    recall  f1-score   support

           0       0.90      0.77      0.83      1035
           1       0.55      0.76      0.64       374

    accuracy                           0.77      1409
   macro avg       0.72      0.77      0.73      1409
weighted avg       0.81      0.77      0.78      1409

ROC-AUC: 0.8449
Confusion Matrix:
 [[802 233]
 [ 91 283]]


In [26]:
import joblib

model_path = os.path.join(r'E:\coding\PROJECTS\customer_churn_prediction\models', 'churn_model_XGB.pkl')
joblib.dump(best_pipe, model_path)

['E:\\coding\\PROJECTS\\customer_churn_prediction\\models\\churn_model_XGB.pkl']